# 🦀 Day 1: Profiling
---

Today we explore **profiling** — measuring before optimizing.

- **Why profiling matters**: Measure before optimizing; avoid premature optimization
- **Types of profiling**: CPU, memory, I/O — each reveals different bottlenecks
- **cargo-flamegraph**: Easy flamegraph generation for visualizing hotspots
- **perf** on Linux: Industry-standard CPU profiler (conceptual for Windows users)
- **std::time::Instant**: Manual timing for quick measurements
- **#[inline]**, **#[cold]**, **#[likely]**: Compiler hints for hot/cold paths

In [ ]:
// Manual timing with std::time::Instant
use std::time::Instant;

fn sum_vec(v: &[i32]) -> i32 {
    v.iter().sum()
}

fn sum_loop(v: &[i32]) -> i32 {
    let mut s = 0;
    for &x in v {
        s += x;
    }
    s
}

let data: Vec<i32> = (0..1_000_000).collect();

let start = Instant::now();
let _ = sum_vec(&data);
let elapsed_vec = start.elapsed();

let start = Instant::now();
let _ = sum_loop(&data);
let elapsed_loop = start.elapsed();

println!("sum_vec: {:?}", elapsed_vec);
println!("sum_loop: {:?}", elapsed_loop);

In [ ]:
// Building a simple benchmarking harness: multiple iterations for stability
use std::time::Instant;

fn bench<F, R>(name: &str, iterations: u32, f: F) -> std::time::Duration
where
    F: Fn() -> R,
{
    let start = Instant::now();
    for _ in 0..iterations {
        std::hint::black_box(f());
    }
    start.elapsed() / iterations
}

// O(n) vs O(n²) vs O(n log n) — timing different algorithms
fn linear(n: usize) -> usize { n }
fn quadratic(n: usize) -> usize { (0..n).flat_map(|i| 0..n).count() }

let n = 100;
let iter_linear = bench("linear", 1000, || linear(n));
let iter_quad = bench("quadratic", 10, || quadratic(n));

println!("Linear O(n) avg: {:?}", iter_linear);
println!("Quadratic O(n²) avg: {:?}", iter_quad);

In [ ]:
// Compiler hints: #[inline], #[cold], #[likely]
// (These are hints — the compiler may ignore them)

#[inline(always)]
fn hot_path(x: i32) -> i32 {
    x * 2
}

#[cold]
fn cold_path() {
    eprintln!("Rare error path");
}

fn maybe_cold(cond: bool) {
    if std::hint::likely(cond) {
        let _ = hot_path(42);
    } else {
        cold_path();
    }
}

maybe_cold(true);
println!("Hints applied — check assembly for effect");

## Tools overview

- **cargo-flamegraph**: `cargo install flamegraph` then `cargo flamegraph` — generates SVG flamegraphs
- **perf** (Linux): `perf record ./target/release/my_bin` then `perf report`
- **valgrind**: Memory errors, cache profiling (Linux/WSL)
- **heaptrack**, **DHAT**: Heap allocation profilers

## 📝 Exercise: Profile two implementations

Implement two versions of "count occurrences of a byte in a slice":
1. Using `iter().filter().count()`
2. Using a manual loop

Use `Instant` and multiple iterations to compare. Which is faster?

In [ ]:
// YOUR CODE HERE
// fn count_byte_iter(slice: &[u8], byte: u8) -> usize { ... }
// fn count_byte_loop(slice: &[u8], byte: u8) -> usize { ... }
// Benchmark both with Instant over 1000 iterations

## 🎯 Key Takeaways

1. **Measure before optimizing** — profiling reveals real bottlenecks
2. **std::time::Instant** is ideal for quick manual timing in Rust
3. **cargo-flamegraph** and **perf** provide visual CPU profiling
4. **#[inline]**, **#[cold]**, **#[likely]** hint the compiler about hot/cold paths
5. Multiple iterations and `black_box` reduce measurement noise
6. Different algorithm complexities (O(n), O(n²)) show up clearly in timing

---
**Tomorrow:** Benchmarking with criterion